#### ¿Cómo optimizar la eficiencia presupuestaria en publicidad?
Con el objetivo de maximizar la rentabilidad del sector de marketing, este proyecto aborda la optimización presupuestaria mediante un estudio descriptivo y exploratorio de la inversión multicanal con periodicidad trimestral (producción del comercial, sueldos de la agencia, edición de videos para redes, etc.). El desafío central no solo reside en identificar los canales con mayor ROI inmediato (eficiencia), sino en comprender la elasticidad y la capacidad de escala de medios masivos como la TV frente a los digitales. A través de evidencia basada en datos, buscamos definir una hoja de ruta financiera que permita proyectar campañas con mayor precisión, logrando una reducción del gasto ineficiente sin comprometer el volumen de crecimiento ni el margen operativo del negocio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output
from scipy.stats import pearsonr, zscore, kurtosis, norm, kstest
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression


df = pd.read_csv("data/advertising_and_sales.csv", index_col="id")

print("Descripción estadística:")
print(df.describe())
print("Primeros registros:")
print(df.head(10))

#### Análisis Descriptivo: descartar y adoptar factores
Como base analítica, a continuación se realiza una visuliazación de análisis multivariable (relacionando valores continuos y categorías), de manera que se puedan extraer resultados a primera vista:

- Hay un claro incremento monetario conforme aumenta el presupuesto en TV y Radio
- Hay una concentración mayor de valores con cierta varianza en la variable TV
- La jerarquía en la variable influencer no influyen en lo absoluto en las ventas

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(20,5))

sns.scatterplot(data=df, x="tv", y="sales", hue="influencer", ax=ax[0])
ax[0].grid("on")
ax[0].set_xlabel("Presupuesto en TV ($)")
ax[0].set_ylabel("ingresos")
ax[0].legend(title="Status de Influencer")

sns.scatterplot(data=df, x="radio", y="sales", hue="influencer", ax=ax[1])
ax[1].grid("on")
ax[1].set_xlabel("Presupuesto en Radio ($)")
ax[1].set_ylabel("")
ax[1].legend(title="Status de Influencer")

sns.scatterplot(data=df, x="social_media", y="sales", hue="influencer", ax=ax[2]) 
ax[2].grid("on")
ax[2].set_xlabel("Presupuesto en Redes Sociales ($)")
ax[2].set_ylabel("")
ax[2].legend(title="Status de Influencer")

fig.suptitle("Relación de inversiones publicitarias e ingresos generados")
plt.subplots_adjust(wspace=0.2)
plt.show()

#### Análisis Exploratorio de Datos: determinar patrones e hipótesis

Seguidamente se explorará el comportamiento de los datos donde el objetivo es descubrir patrones, relaciones y estructuras que guíen la implementación de un modelo particular como solución. En un caso de regresión lineal con resultados fiables, se deben cumplir los siguientes supuestos fundamentales:

Linealidad: La relación entre la variable independiente (X) y la variable dependiente (Y) debe ser lineal. Esto significa que el cambio en (Y) por cada unidad de cambio en (X) es constante.

Homocedasticidad: La varianza de los errores debe ser constante para todos los niveles de las variables independientes. Si la varianza cambia(por ejemplo, aumenta al aumentar X), existe heterocedasticidad.

Normalidad de los residuos: Para propósitos de inferencia (pruebas de hipótesis y cálculos de intervalos de confianza), los residuos del modelo deben seguir una distribución normal con media cero.

Ausencia de valores atípicos (Outliers): Los datos no deben contener valores extremos influyentes que distorsionen significativamente la pendiente de la línea de regresión.

In [ ]:
corr_tv, _ = pearsonr(df["tv"], df["sales"])
corr_radio, _ = pearsonr(df["radio"], df["sales"])

df_zscore = df.loc[:, ["tv","radio"]]

df_zscore["tv_zscore"] = abs(zscore(df["tv"]))
df_zscore["radio_zscore"] = abs(zscore(df["radio"]))

outliers_tv = df_zscore.loc[df_zscore["tv_zscore"] > 3, "tv_zscore"].shape[0]
outliers_radio = df_zscore.loc[df_zscore["radio_zscore"] > 3, "radio_zscore"].shape[0]

curtosis_tv = kurtosis(df["tv"])
curtosis_radio = kurtosis(df["radio"])

data = {
   "TV": [round(corr_tv,2), outliers_tv, round(curtosis_tv,1)],
   "Radio": [round(corr_radio,2), outliers_radio, round(curtosis_radio,1)]
}

df_measures  = pd.DataFrame(data, index=["correlacion (sales)", "outliers", "curtosis"])

print(df_measures)

#### Pruebas de normalidad
Para garantizar la fiabilidad del modelo de regresión, se complementó el análisis visual con pruebas de hipótesis robustas que permiten diagnosticar la distribución de las variables (Inversión en TV e Ingresos) con mayor precisión, reduciendo el riesgo de sesgos de confirmación.

Shapiro-Wilk: Evalúa la normalidad mediante la correlación entre los datos observados y sus correspondientes valores teóricos en una distribución normal (puntuaciones Z). Es considerada una de las pruebas más potentes para detectar desviaciones de la normalidad en muestras moderadas.

Kolmogorov-Smirnov: Cuantifica la distancia máxima entre la función de distribución acumulada (CDF) de la muestra y la de una distribución normal teórica. Es especialmente útil para identificar discrepancias en la forma general de la distribución.

Ambas pruebas utilizan el valor p (p-value) para validar la naturaleza de los datos: Bajo un nivel de significancia del 5% (alpha =0.05)), un valor p inferior a este umbral nos obliga a rechazar la normalidad

In [ ]:
_, p_value_var_x = kstest(df["tv"], "norm")
_, p_value_var_y = kstest(df["sales"], "norm")

fig, ax = plt.subplots(2,2, figsize=(20,6))

sns.histplot(data=df, x="tv", kde=True, ax=ax[0,0])
ax[0,0].set_title("Histograma en Variable X")
ax[0,0].set_xlabel("")
ax[0,0].set_ylabel("")

sns.boxplot(data=df, x="tv", ax=ax[0,1])
ax[0,1].set_title("Gráfico de caja en Variable X")
ax[0,1].set_xlabel("")
ax[0,1].set_ylabel("")

sns.histplot(data=df, x="sales", kde=True, ax=ax[1,0])
ax[1,0].set_title("Histograma en Variable Y")
ax[1,0].set_xlabel("")
ax[1,0].set_ylabel("")

sns.boxplot(data=df, x="sales", ax=ax[1,1])
ax[1,1].set_title("Gráfico de caja en Variable Y")
ax[1,1].set_xlabel("")
ax[1,1].set_ylabel("")

plt.subplots_adjust(hspace=0.5)
plt.suptitle("Figura de distribución (variable X vs variable Y)")
plt.show()

#### Linear Regression: modelado segmentado por mayor rentabilidad
Tras confirmar la dependencia estadística entre variables, procedemos a validar al canal TV como el estímulo principal del volumen de ventas. Para ello, se implementa un modelado ajustado estratégicamente hacia la rentabilidad: en lugar de promediar el desempeño general, se entrena el algoritmo sobre la 'Frontera de Eficiencia' (cuantil 12 superior de ROI), este enfoque permite decantar las mejores prácticas históricas y transformar la regresión lineal en un marco de referencia optimizado. El resultado es una base científica que ratifica la potencia del canal y justifica la redirección del gasto hacia escenarios de máxima rentabilidad.

In [5]:
X_train = df_model["tv"].values.reshape((-1, 1))
y_train = df_model["sales"]

model_frontier = LinearRegression()
model_frontier.fit(X_train, y_train)

y_pred_train = model_frontier.predict(X_train)

r2 = r2_score(y_train, y_pred_train)
mae = mean_absolute_error(y_train, y_pred_train)
rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))

print("--- RESULTADOS DE VALIDACIÓN DEL MODELO (EFICIENTE) ---")
print(f"Coeficiente de Determinación (R²): {round(r2, 4)}")
print(f"Error Absoluto Medio (MAE): ${round(mae, 2)}")
print(f"Raíz del Error Cuadrático Medio (RMSE): ${round(rmse, 2)}")
print("-" * 50)
print(f"Interpretación: El modelo explica el {round(r2*100, 2)}% de la variación en ventas.")
if r2 > 0.80:
    print("Estado: MODELO VALIDADO PARA PRODUCCIÓN (DASHBOARD)")
else:
    print("Estado: REVISAR SEGMENTACIÓN - Precisión por debajo del umbral objetivo.")


#### Dashboard de optimización estratégica basado en la frontera de eficiencia y el modelo lineal

In [ ]:
def get_elasticities(data):
    df_log = np.log(data[["tv", "radio", "social_media", "sales"]] + 1)
    model = LinearRegression()
    model.fit(df_log[["tv", "radio", "social_media"]], df_log["sales"])
    return dict(zip(["TV", "Radio", "Social Media"], model.coef_))

elasticities = get_elasticities(df)
mean_margin = df["sales"].mean() - (df["tv"].mean() + df["radio"].mean() + df["social_media"].mean())

app = dash.Dash(__name__)

app.layout = html.Div(id="body", className="e2_body", children=[
    html.H1("Estrategia de Inversión: del volumen a la rentabilidad", className="e2_title"),

    html.Div(className="e2_div_stats", children=[
        html.Div([html.H3("Margen Neto Promedio"), html.P(f"${round(mean_margin, 2)}", style={"font-weight":"bold","font-size":"1.2em"})], className="e2_stats"),
        html.Div([html.H3("Elasticidad TV (Dominancia)"), html.P(f"{round(elasticities["TV"], 2)}", style={"font-weight":"bold","font-size":"1.2em"})], className="e2_stats"),
        html.Div([html.H3("Elasticidad Radio (Oportunidad)"), html.P(f"{round(elasticities["Radio"], 2)}", style={"font-weight":"bold","font-size":"1.2em"})], className="e2_stats"),
    ]),

    html.Div(id="dashboard", className="e2_dashboard", children=[
         html.Div(className="e2_column_1", children=[
             html.Div(className="e2_div_graphs", children=[
                 dcc.Graph(id="graph-pie", className="e2_graphs", figure={}),
                 dcc.Graph(id="graph-bar", className="e2_graphs", figure={})
             ]),

             html.Div(className="e2_div_slider", children=[
                html.Label("Simulador de Rebalanceo: Mover presupuesto de TV a Radio (%)", className="e2_label"),
                dcc.Slider(
                    id="rebalance-slider", 
                    min=0, max=30, step=5, value=0, 
                    marks={i: {"label": f"{i}%", "style": {"color": "white"}} for i in range(0, 31, 5)}
                )
            ])
        ]),

        html.Div(className="e2_column_2", children=[
           dcc.Graph(id="graph-scatter"),
           html.Div(id="text-resolution", className="e2_text_resolution")
        ])
    ])
])


@app.callback(
    [Output(component_id="graph-pie",component_property="figure"),
    Output(component_id="graph-bar",component_property="figure"),
    Output(component_id="graph-scatter",component_property="figure"),
    Output(component_id="text-resolution",component_property="children")],
   [Input(component_id="rebalance-slider",component_property="value")]
)

def update_strategy(rebalance_pct):
    cost_tv_orig = df["tv"].mean()
    money_change = cost_tv_orig * (rebalance_pct / 100)

    piechart = go.Figure(data=[go.Pie(labels=["TV", "Radio", "RRSS"],
                               values=[df["tv"].mean(), df["radio"].mean(), df["social_media"].mean()])])
    piechart.update_layout(title="Distribución Actual del Gasto")

    barchart = go.Figure([go.Bar(x=list(elasticities.keys()), y=list(elasticities.values()), marker_color="indigo")])
    barchart.update_layout(title="Elasticidad: Sensibilidad de Ventas al +1% Gasto", yaxis_title="Impacto % en Ventas")

    df["ROI"] = (df["sales"] - df["tv"]) / df["tv"]
    df_efficient = df[df["ROI"] >= df["ROI"].quantile(0.12)]

    scatter = go.Figure()
    scatter.add_trace(go.Scatter(x=df["tv"], y=df["sales"], mode="markers", name="Campaña Estándar", opacity=0.4))
    scatter.add_trace(go.Scatter(x=df_efficient["tv"], y=df_efficient["sales"], mode="markers", name="Frontera de Eficiencia", marker=dict(color="green", size=8)))
    scatter.update_layout(title="Identificación de la Frontera de Eficiencia", xaxis_title="Gasto TV", yaxis_title="Ventas")

    sales_est = df["sales"].mean() * (1 + (rebalance_pct/100 * elasticities["Radio"]))
    text = [
        html.H4("Resolución Estratégica:"),
        html.P(f"Al rebalancear un {rebalance_pct}% de TV hacia Radio, estás moviendo ${round(money_change, 2)} a un canal con mayor retorno marginal."),
        html.P(f"Estimación: Las ventas podrían optimizarse un {round(rebalance_pct * elasticities["Radio"], 2)}% manteniendo el presupuesto constante.")
    ]

    return piechart, barchart, scatter, text

if __name__ == "__main__":
    app.run(debug=False)

#### Frontera de eficiencia inversora
El análisis de la frontera de eficiencia revela una oportunidad de optimización crítica: existen escenarios históricos donde se alcanzaron niveles de ingresos similares con presupuestos significativamente menores, lo que demuestra una ineficiencia en el gasto actual. Mientras que la TV garantiza el volumen, el modelo identifica que hemos superado el punto de rendimientos decrecientes en ese canal. La resolución del problema no reside en aumentar el gasto, sino en un rebalanceo estratégico del mix de medios. La propuesta concreta es capturar el alto potencial de la Radio (~15%) y canales digitales, donde el costo marginal es menor. Al redistribuir una fracción del presupuesto de TV hacia estos canales de mayor agilidad, es posible maximizar el margen neto y la velocidad de retorno sin sacrificar el alcance global.